In [1]:
# Cell 1 - imports and paths
import os
import csv
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

# Paths (adjust if your repo layout differs)
RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

TRAIN_RAW = os.path.join(RAW_DIR, 'train.csv')
TEST_RAW = os.path.join(RAW_DIR, 'test.csv')
TRAIN_OUT = os.path.join(PROCESSED_DIR, 'train_features_text.csv')
TEST_OUT  = os.path.join(PROCESSED_DIR, 'test_features_text.csv')

print("Paths set.")
print("TRAIN_RAW:", TRAIN_RAW)
print("TEST_RAW: ", TEST_RAW)
print("TRAIN_OUT:", TRAIN_OUT)
print("TEST_OUT: ", TEST_OUT)


Paths set.
TRAIN_RAW: ..\data\raw\train.csv
TEST_RAW:  ..\data\raw\test.csv
TRAIN_OUT: ..\data\processed\train_features_text.csv
TEST_OUT:  ..\data\processed\test_features_text.csv


In [2]:
# Cell 2 - feature extraction utilities

# units of measure pattern (used to differentiate size vs pack-count)
UNITS_OF_MEASURE = (
    r'oz|ounce|o z|lb|pound|g|gram|kg|kilo|mg|milligrams|'
    r'ml|milliliter|milliliters|l|ltr|liter|liters|cl|'
    r'gal|gallon|qt|quart|fl oz|fluid ounce'
)

def extract_item_size_and_unit(text):
    """Return (quantity: float or np.nan, unit: str or None) from text"""
    text = '' if text is None else str(text)
    pattern = re.compile(r'(?P<quantity>\d*\.?\d+)\s*(?P<unit>' + UNITS_OF_MEASURE + r')', re.IGNORECASE)
    m = pattern.search(text)
    if not m:
        return np.nan, None
    try:
        return float(m.group('quantity')), m.group('unit').strip().lower()
    except:
        return np.nan, None

def extract_pack_count_final_v2(text):
    """
    Robust pack-count extraction. Returns float pack count (>=1).
    If nothing found, returns 1.0 (single item).
    """
    text = '' if text is None else str(text).lower().replace('×', 'x').replace(',', '')
    text = re.sub(r'[^\x00-\x7f]', ' ', text)

    # word maps
    WORD_NUM_MAP = {"half dozen": 6, "dozen": 12, "single": 1, "pair": 2, "double": 2, "triple": 3, "trio": 3}

    for k, v in WORD_NUM_MAP.items():
        if k in text:
            return float(v)

    # multiplication patterns, e.g., "6 x 12", "12x6"
    m = re.search(r'\b(\d{1,4}(?:\.\d+)?)\s*[x\*]\s*(\d{1,4}(?:\.\d+)?)\b', text)
    if m:
        try:
            return float(float(m.group(1)) * float(m.group(2)))
        except:
            pass

    # patterns like "12 pack", "pack of 12", "(pack of 12)"
    patterns = [
        r'\b(?:pack(?:age)?|case|box|set|bundle|combo|pk|pks|ct|count|pcs|pieces|pkg)\b[^\d]{0,6}?(\d{1,4}(?:\.\d+)?)',
        r'\b(\d{1,4}(?:\.\d+)?)(?:\s|-)?(?:pack|pk|pks|packs|pkg|ct|count|pcs|pieces)\b',
        r'\( ?(?:pack(?: of)?|pk|pks|packs|count|ct) ?(\d{1,4}) ?\)'
    ]
    for p in patterns:
        m = re.search(p, text)
        if m:
            try:
                val = float(m.group(1))
                if val >= 1:
                    return val
            except:
                pass

    return 1.0


In [3]:
# Cell 3 - load train robustly and apply feature extraction

import sys
import csv

def robust_read_csv(path):
    """Read CSV with python engine and tolerant handling for messy quoted fields."""
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return pd.read_csv(path, engine='python', encoding='utf-8', quoting=csv.QUOTE_MINIMAL, on_bad_lines='warn')

# Load train (fallback to dummy if missing)
try:
    train_df = robust_read_csv(TRAIN_RAW)
    print("Loaded train.csv shape:", train_df.shape)
except FileNotFoundError:
    print("TRAIN_RAW not found. Creating dummy train_df for demonstration.")
    train_df = pd.DataFrame({
        'sample_id': [9259, 292686],
        'catalog_content': [
            "Item Name: Member's Mark, Basil, 6.25 oz",
            "Item Name: kedem Sherry Cooking Wine, 12.7 Ounce - 12 per case."
        ],
        'image_link': ['link1', 'link2'],
        'price': [18.5, 66.49]
    })
    print("Dummy train_df created.")

# Ensure required columns exist
for req in ['sample_id', 'catalog_content', 'image_link']:
    if req not in train_df.columns:
        train_df[req] = np.nan

# Apply size & unit extraction
temp = train_df['catalog_content'].apply(extract_item_size_and_unit).apply(pd.Series)
train_df['item_size'] = temp[0]
train_df['unit'] = temp[1]

# Pack count
train_df['pack_count'] = train_df['catalog_content'].apply(extract_pack_count_final_v2)

# Default fills
train_df['item_size'].fillna(1.0, inplace=True)
train_df['unit'].fillna('unit', inplace=True)
train_df['pack_count'].fillna(1.0, inplace=True)

# total_quantity
train_df['total_quantity'] = train_df['item_size'] * train_df['pack_count']

# normalize unit strings (a short map)
unit_replacement_map = {
    'oz': 'ounce', 'o z': 'ounce', 'ounces': 'ounce',
    'milliliter': 'ml', 'milliliters': 'ml', 'mls': 'ml',
    'gram': 'g', 'grams': 'g', 'kilogram': 'kg', 'kilograms': 'kg',
    'pound': 'lb', 'pounds': 'lb', 'lbs': 'lb',
    'piece': 'pc', 'pieces': 'pc'
}
train_df['unit'] = train_df['unit'].replace(unit_replacement_map)

print("Train features applied. Columns now include:", list(train_df.columns))
display(train_df[['sample_id','catalog_content','item_size','unit','pack_count','total_quantity']].head())


Loaded train.csv shape: (75000, 4)
Train features applied. Columns now include: ['sample_id', 'catalog_content', 'image_link', 'price', 'item_size', 'unit', 'pack_count', 'total_quantity']


C:\Users\nikk6\AppData\Local\Temp\ipykernel_21868\2177423818.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['item_size'].fillna(1.0, inplace=True)
C:\Users\nikk6\AppData\Local\Temp\ipykernel_21868\2177423818.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For ex

,sample_id,catalog_content,item_size,unit,pack_count,total_quantity
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",12.00,ounce,6.0,72.00
1,198967,"Item Name: Salerno Cookies, The Original Butte...",8.00,ounce,4.0,32.00
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",1.90,ounce,1.0,1.90
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,11.25,ounce,1.0,11.25
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",12.70,ounce,1.0,12.70


In [12]:
# Cell X.1 - Priority-1 diagnostics: skew, top percentiles, duplicate checks
import numpy as np
import pandas as pd

def priority1_diagnostics(df, name="train_df"):
    print(f"=== Diagnostics for {name} ===")
    if 'price' not in df.columns:
        print("No 'price' column in this DataFrame.")
        return
    prices = df['price'].dropna().astype(float)
    pct = [50, 75, 90, 95, 99, 99.5, 99.9]
    quantiles = prices.quantile([q/100 for q in pct]).to_dict()
    for p in pct:
        print(f"  {p}th pct: {quantiles[p/100] if (p/100) in quantiles else prices.quantile(p/100)}")
    print("Median:", prices.median(), "Mean:", prices.mean(), "Std:", prices.std(), "Max:", prices.max())
    print("Max / Median ratio:", prices.max() / (prices.median() + 1e-9))
    # price_per_unit stability check (if total_quantity exists)
    if 'total_quantity' in df.columns:
        pq = (df['price'] / df['total_quantity']).replace([np.inf, -np.inf], np.nan).dropna()
        print(" price_per_unit stats - median:", pq.median(), "std:", pq.std(), "unique per sample (sample):")
        print("  sample of price_per_unit values:", pq.sample(min(len(pq),5), random_state=1).tolist())
    # duplicates by catalog_content
    if 'catalog_content' in df.columns:
        dup_counts = df.groupby('catalog_content')['price'].nunique()
        many_prices = (dup_counts > 1).sum()
        print("Rows with same catalog_content but >1 unique price:", many_prices)
    # show top extreme rows
    print("\nTop 8 most expensive rows:")
    display(df.sort_values('price', ascending=False).head(8))
    print("\nBottom 8 (lowest non-zero) rows:")
    display(df[df['price']>0].sort_values('price', ascending=True).head(8))

# Run diagnostics on your in-memory train_df and optionally test_df
priority1_diagnostics(train_df, "train_df")
# if you have a test_df
# priority1_diagnostics(test_df, "test_df")


=== Diagnostics for train_df ===
  50th pct: 14.0
  75th pct: 28.625
  90th pct: 52.30100000000006
  95th pct: 75.71100000000006
  99th pct: 145.25029999999984
  99.5th pct: 183.70050000000046
  99.9th pct: 322.1217400000067
Median: 14.0 Mean: 23.647654 Std: 33.37693218315573 Max: 2796.0
Max / Median ratio: 199.7142857000204
 price_per_unit stats - median: 1.143125 std: 35.296207982224146 unique per sample (sample):
  sample of price_per_unit values: [3.67625, 3.0, 0.433125, 36.208333333333336, 22.425]
Rows with same catalog_content but >1 unique price: 67

Top 8 most expensive rows:


,sample_id,catalog_content,image_link,price,item_size,unit,pack_count,total_quantity,unit_target_encoded
58617,229126,Item Name: 4Patriots 1-Year Survival Food Kit:...,https://m.media-amazon.com/images/I/81WiVwz7Kk...,2796.00,1.00,unit,3.0,3.00,22.871360
32153,134909,Item Name: Royal Amber Osetra Caviar - Russian...,https://m.media-amazon.com/images/I/31fGy7wOV7...,1280.00,16.00,ounce,1.0,16.00,19.726170
22839,62251,Item Name: 495 days proteins meal 33 bottles x...,https://m.media-amazon.com/images/I/71z7pFftrI...,1188.00,3.90,g,2.0,7.80,26.132965
46301,195806,Item Name: China Beauty Rings Tea (Loose) (8 o...,https://m.media-amazon.com/images/I/81xJvPGH3Y...,1010.54,8.00,ounce,2.0,16.00,19.839793
68388,236048,"Item Name: Organic Pistachios Shelled Raw, 30 ...",https://m.media-amazon.com/images/I/81eC194y1z...,921.50,30.00,lb,1.0,30.00,42.157091
72853,262694,Item Name: Numanna Family Pack Bucket 432 Serv...,https://m.media-amazon.com/images/I/812s9xubmA...,779.25,1.00,unit,2.0,2.00,23.248667
73067,190026,"Item Name: Crystal Geyser Pallet Of 84 Cases, ...",https://m.media-amazon.com/images/I/A1Th9c7NqN...,739.99,16.90,ounce,1.0,16.90,19.839793
45580,272914,"Item Name: Topps Juicy Drop Pop, 18 count per ...",https://m.media-amazon.com/images/I/71pIYhTtyo...,691.16,0.92,ounce,16.0,14.72,19.674755



Bottom 8 (lowest non-zero) rows:


,sample_id,catalog_content,image_link,price,item_size,unit,pack_count,total_quantity,unit_target_encoded
32750,212063,Item Name: Frank's RedHot Squeeze Sriracha Sau...,https://m.media-amazon.com/images/I/71ILSRoj02...,0.13,1.70,fl oz,24.0,40.80,16.568884
1774,261551,"Item Name: Merci European Chocolates, 7 Ounce ...",https://m.media-amazon.com/images/I/71sxsD0SEz...,0.13,7.00,ounce,10.0,70.00,19.674755
66444,144562,"Item Name: Maruchan Ramen Noodle Soup, Beef, 3...",https://m.media-amazon.com/images/I/91LcOkLiQ1...,0.30,3.00,ounce,1.0,3.00,19.839793
6209,144673,Item Name: Maruchan Creamy Chicken Ramen Noodl...,https://m.media-amazon.com/images/I/918MLnfD3L...,0.33,3.00,ounce,1.0,3.00,19.674755
65384,298503,Item Name: Kool Aid Grape - 3.9g\nValue: 0.14\...,https://m.media-amazon.com/images/I/5172VLcXmV...,0.36,3.90,g,1.0,3.90,27.053497
5982,25417,"Item Name: Maruchan Ramen Beef, 0.1875-Ounce P...",https://m.media-amazon.com/images/I/81+2i1EwSm...,0.36,1.00,unit,24.0,24.00,22.871360
31076,86997,Item Name: Kool-aid Watermelon Unsweetened 15 ...,https://m.media-amazon.com/images/I/71vAPJjkAi...,0.36,0.15,ounce,1.0,0.15,19.734479
74116,173321,"Item Name: Kool Aid Watermelon Drink Mix, Make...",https://m.media-amazon.com/images/I/81R1q61kt7...,0.37,2.00,quart,192.0,384.00,32.273672


In [13]:
# Cell X.2 - create price_per_unit and fill safety defaults
import numpy as np

# Ensure total_quantity exists and > 0
for df, name in [(train_df, 'train_df'), (test_df, 'test_df')]:
    if 'total_quantity' not in df.columns:
        raise KeyError("total_quantity missing — run feature extraction first.")
    # replace zeros or negatives with 1 to avoid division problems (logically: you can choose other handling)
    df['total_quantity'] = df['total_quantity'].apply(lambda x: x if (pd.notna(x) and x>0) else 1.0)

# Price per unit
if 'price' in train_df.columns:
    train_df['price_per_unit'] = train_df['price'] / train_df['total_quantity']
else:
    train_df['price_per_unit'] = np.nan
test_df['price_per_unit'] = np.nan  # will be populated from train mapping later if needed

# Show quick stats
print("price_per_unit stats (train):")
print(train_df['price_per_unit'].describe().to_string())
print("\nSample price_per_unit values:")
display(train_df[['catalog_content','price','total_quantity','price_per_unit']].sample(min(6,len(train_df)), random_state=1))


price_per_unit stats (train):
count    7.500000e+04
mean     7.063605e+00
std      3.527888e+01
min      3.377526e-07
25%      2.832420e-01
50%      1.152706e+00
75%      5.460417e+00
max      3.361500e+03

Sample price_per_unit values:


,catalog_content,price,total_quantity,price_per_unit
11591,"Item Name: Dill Rooibos Tea (50 tea bags, ZIN:...",86.02,3.0,28.673333
52020,"Item Name: 7 UP A&W Cream Soda Soft Drink, 20-...",2.39,24.0,0.099583
34666,Item Name: Maldon Salt Smoked Sea Salt 125G Bu...,17.99,250.0,0.071960
22169,Item Name: D.ivina Organic Pitted Kalamata Oli...,54.99,36.0,1.527500
23049,Item Name: Roasted Salted Corn Nuts Snack with...,36.99,4.0,9.247500
26989,Item Name: Stroop Club Caramel Delights Cookie...,39.99,24.0,1.666250


In [14]:
# Cell X.3 - Outlier handling: choose 'cap' or 'remove' and percentile threshold
strategy = 'cap'     # 'cap'  => winsorize top X percentile; 'remove' => drop rows above percentile
percentile = 99.9     # common choices: 99, 99.5, 99.9

def apply_outlier_control(df, col='price', strategy='cap', top_percentile=99.9):
    df = df.copy()
    if col not in df.columns:
        return df
    cutoff = np.nanpercentile(df[col].dropna(), top_percentile)
    before = df.shape[0]
    if strategy == 'cap':
        # cap values above cutoff
        df[col] = df[col].clip(upper=cutoff)
        print(f"Capped {col} at {top_percentile}th percentile value = {cutoff:.4f}")
    elif strategy == 'remove':
        df = df[df[col] <= cutoff].reset_index(drop=True)
        print(f"Removed rows with {col} > {top_percentile}th percentile ({cutoff:.4f}). Removed: {before - df.shape[0]} rows")
    else:
        raise ValueError("strategy must be 'cap' or 'remove'")
    return df

# Apply on a copy so you can inspect before overwriting original train_df
train_df_pre = train_df.copy()
train_df_after = apply_outlier_control(train_df_pre, col='price', strategy=strategy, top_percentile=percentile)

print("Before shape:", train_df_pre.shape, "After shape:", train_df_after.shape)
print("Price stats after outlier handling:")
display(train_df_after['price'].describe())
# If you like the change, assign it back:
# train_df = train_df_after


Capped price at 99.9th percentile value = 322.1217
Before shape: (75000, 10) After shape: (75000, 10)
Price stats after outlier handling:


count    75000.000000
mean        23.471484
std         29.553979
min          0.130000
25%          6.795000
50%         14.000000
75%         28.625000
max        322.121740
Name: price, dtype: float64

In [15]:
# Cell X.4 - create model target (pick one approach)
# options: target_mode = 'log_price' OR 'log_price_per_unit'
target_mode = 'log_price'   # or 'log_price_per_unit'

if target_mode == 'log_price':
    if 'price' not in train_df.columns:
        raise KeyError("price missing in train_df")
    train_df['model_target'] = np.log1p(train_df['price'].astype(float))
    reconstruct_from_pred = lambda pred: np.expm1(pred)   # invert model_target predictions -> price
    print("Using log1p(price) as model target.")
elif target_mode == 'log_price_per_unit':
    if 'price_per_unit' not in train_df.columns:
        raise KeyError("price_per_unit missing in train_df")
    train_df['model_target'] = np.log1p(train_df['price_per_unit'].astype(float))
    # to reconstruct price: price = expm1(predicted_unit_price) * total_quantity
    reconstruct_from_pred = lambda pred, qty: np.expm1(pred) * qty
    print("Using log1p(price_per_unit) as model target (remember to multiply by total_quantity at inference).")
else:
    raise ValueError("target_mode must be 'log_price' or 'log_price_per_unit'")

print("model_target stats:")
display(train_df['model_target'].describe())


Using log1p(price) as model target.
model_target stats:


count    75000.000000
mean         2.739217
std          0.942032
min          0.122218
25%          2.053483
50%          2.708050
75%          3.388619
max          7.936303
Name: model_target, dtype: float64

In [16]:
# Cell X.5 - simple label cleaning & duplicate inspection
# 1) Remove exact duplicates (same sample_id) keeping the first
if 'sample_id' in train_df.columns:
    dup_ids = train_df['sample_id'].duplicated().sum()
    if dup_ids:
        print(f"Found {dup_ids} duplicated sample_id rows. Dropping duplicates (keeping first).")
        train_df.drop_duplicates(subset=['sample_id'], keep='first', inplace=True)
    else:
        print("No sample_id duplicates.")

# 2) Check rows with identical catalog_content but wildly different prices (potential label noise)
if 'catalog_content' in train_df.columns:
    price_spread = train_df.groupby('catalog_content')['price'].agg(['count','min','max'])
    noisy = price_spread[(price_spread['count']>1) & ((price_spread['max'] / (price_spread['min'] + 1e-9)) > 4)].sort_values('max', ascending=False)
    print("Catalog entries with >1 listing and >4x max/min price (top 10):")
    display(noisy.head(10))
    if len(noisy) > 0:
        print("Consider inspecting these catalog_content values manually; they may be mis-parsed or multi-SKU rows.")

# 3) Optional: drop rows with price <= 0 or extreme negative values
bad_price_rows = ((train_df['price'] <= 0) | (train_df['price'].isna()))
if bad_price_rows.any():
    print("Dropping rows with non-positive or missing price:", bad_price_rows.sum())
    train_df = train_df[~bad_price_rows].reset_index(drop=True)
else:
    print("No non-positive prices found.")


No sample_id duplicates.
Catalog entries with >1 listing and >4x max/min price (top 10):


,count,min,max
catalog_content,,,
"Item Name: Blooms2Door 100 Red Roses (Farm-Fresh, Long Stem - 50cm) - Farm Direct Wholesale Fresh Flowers\nValue: 100.0\nUnit: Count\n",2,71.670,320.450
"Item Name: Rose’s Sweetened Lime Syrup 12oz Bottle | Perfect for Cocktails, Beverages, and Mixers\nBullet Point 1: Sweetened Lime: Delivers a tangy, authentic lime flavor, made with real lime juice and perfectly sweetened for ideal mixability in your favorite cocktails, spirits, and non-alcoholic drinks\nBullet Point 2: Trusted for Over a Century: Rose’s has been the go-to brand for premium mixers for more than 100 years, earning the trust of bartenders and cocktail enthusiasts alike.\nBullet Point 3: Exceptional Quality: Renowned for its consistent, high-quality mixers, Rose’s ensures your cocktails are always perfectly balanced and flavorful.\nBullet Point 4: Versatile Flavor Range: Available in a variety of flavors, including Lime, Grenadine, Peach, Blueberry, Sweet and Sour, Strawberry, and Simple Syrup, to complement a wide array of cocktail recipes.\nBullet Point 5: Perfect for Classic & Modern Cocktails: Whether you're mixing timeless drinks or creating new concoctions, Rose’s mixers add a touch of excellence to every beverage.\nValue: 12.0\nUnit: Fl Oz\n",2,11.980,60.880
"Item Name: V8 Spicy Hot 100% Vegetable Juice, 11.5 fl oz Can (24 Pack)\nBullet Point 1: Twenty-four (24) 11.5 fl oz single-serve cans of V8 Spicy Hot 100% Vegetable Juice\nBullet Point 2: A satisfying alternative to other juices made with concentrated tomato juice along with the juices of seven other vegetables\nBullet Point 3: Each 11.5 fl oz can of this 100% juice contains 2.5 servings of vegetables and is an excellent source of Vitamins A and C\nBullet Point 4: Gluten free and non GMO veggie juice with no sugar added* (*Not a low calorie food; see nutrition panel for sugar and calorie content)\nBullet Point 5: An easy way to help get your daily recommended veggies; enjoy it as a breakfast drink, afternoon snack, or post workout drink\nValue: 276.0\nUnit: Fl Oz\n",2,9.150,46.990
Item Name: TAZO TEA\nValue: nan\nUnit: None\n,2,4.490,26.990
"Item Name: French's White Cheddar Crispy Fried Onions, 6 oz\nBullet Point 1: Made with real onions\nBullet Point 2: White cheddar and onion-flavored topping with a crispy, crunchy texture\nBullet Point 3: Resealable package for freshness\nBullet Point 4: Kosher certified; product of the USA\nBullet Point 5: Add a pop of cheesy, fried onion taste and texture to salads, soups and loaded mashed potatoes\nValue: 6.0\nUnit: Ounce\n",3,5.190,26.960
"Item Name: McCormick Golden Dipt Cracker Meal Seafood Fry Mix, 10 oz (Pack of 8)\nBullet Point 1: Wheat flour-based fry mix gives seafood a crumb coating that seals in juices\nBullet Point 2: 3 easy steps: lightly moisten fish, coat with Cracker Meal, then fry\nBullet Point 3: Tasty with tilapia, shrimp, scallops, shucked oysters and soft-shell crabs\nBullet Point 4: Substitute Cracker Meal for bread crumbs in meat loaf or meat balls\nBullet Point 5: Dip fish in a mixture of 3 tbsp milk and 2 beaten eggs for a thicker coating\nValue: 80.0\nUnit: Ounce\n",2,5.635,26.800
"Item Name: Concord Foods Banana Smoothie Mix - Fruit Flavor with No Artificial Flavors, Colors, or Preservatives - Ideal for Fresh Fruit Smoothies - 2 oz Pouch for Healthy Smoothies Pack of 12\nBullet Point 1: Banana Goodness for Vibrant Creations: Craft vibrant smoothies and yogurt cups bursting with banana goodness using our Banana Smoothie Mix. Get your daily dose of fruits conveniently with this blend, perfect for a quick and nutritious boost.\nBullet Point 2: Real Fruit Smoothie Enhancement: Elevate your smoothie experience with our fruit smoothie mix. Packed with real fruit flavor and free from artificial colors, flavors, and preservatives, it's the perfect choice for health-conscious consumers seeking a refreshing and nutritious treat.\nBullet Point 3: No Artificial Flavors: Savor the pure taste of real fruit goodnes

Consider inspecting these catalog_content values manually; they may be mis-parsed or multi-SKU rows.
No non-positive prices found.


In [17]:
# Cell X.6 - finalize for modeling and ensure unit_target_encoded exists (fallback)
import numpy as np

# If you chose log_price_per_unit you will predict per-unit on test; else you predict price directly.
if target_mode == 'log_price_per_unit':
    # create test.model_target placeholder (if test has price not known, it's fine)
    test_df['model_target'] = np.nan
    print("test_df model_target left NaN (will predict unit price then multiply by total_quantity).")

# Ensure unit_target_encoded exists (if you computed earlier CV encoding, this will keep it)
if 'unit_target_encoded' not in train_df.columns:
    # fallback: simple mean encode (non-CV) with smoothing small alpha
    alpha = 10
    agg = train_df.groupby('unit')['price'].agg(['mean','count'])
    global_mean = train_df['price'].mean()
    agg['smooth'] = (agg['count'] * agg['mean'] + alpha * global_mean) / (agg['count'] + alpha)
    train_df['unit_target_encoded'] = train_df['unit'].map(agg['smooth']).fillna(global_mean)
    test_df['unit_target_encoded']  = test_df['unit'].map(agg['smooth']).fillna(global_mean)
    print("Created fallback unit_target_encoded from full-train smoothed means.")
else:
    # if unit_target_encoded exists, map train->test mapping for consistency
    # For test, use full-train mean mapping
    train_map = train_df.groupby('unit')['price'].mean()
    test_df['unit_target_encoded'] = test_df['unit'].map(train_map).fillna(train_df['price'].mean())
    print("Ensured test_df.unit_target_encoded mapped from train means.")

# Final quick shapes
print("Final shapes: train_df:", train_df.shape, " test_df:", test_df.shape)
print("Saved columns on train (sample):", train_df.columns.tolist()[:20])


Ensured test_df.unit_target_encoded mapped from train means.
Final shapes: train_df: (75000, 11)  test_df: (75000, 9)
Saved columns on train (sample): ['sample_id', 'catalog_content', 'image_link', 'price', 'item_size', 'unit', 'pack_count', 'total_quantity', 'unit_target_encoded', 'price_per_unit', 'model_target']


In [18]:
# Cell 4 - cross-validated target encoding for 'unit' -> 'unit_target_encoded'

target_col = 'price'
cat_col = 'unit'
cv_splits = 5

# create column
train_df['unit_target_encoded'] = np.nan

if target_col not in train_df.columns:
    print("Warning: 'price' column not found in train. unit_target_encoded will be NaN.")
else:
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        tr = train_df.iloc[tr_idx]
        val = train_df.iloc[val_idx]

        # compute mean price per unit on training fold
        fold_map = tr.groupby(cat_col)[target_col].mean()
        # map to validation rows
        mapped = val[cat_col].map(fold_map)
        train_df.loc[train_df.index[val_idx], 'unit_target_encoded'] = mapped.values

    # fill leftover NaNs with global mean price
    global_mean = train_df[target_col].mean()
    train_df['unit_target_encoded'].fillna(global_mean, inplace=True)
    print("Cross-validated unit_target_encoded computed. Global mean fill:", global_mean)

# quick stats
print(train_df[['unit','unit_target_encoded']].groupby('unit').agg(['count','mean']).head().to_string())


Cross-validated unit_target_encoded computed. Global mean fill: 23.647654
            unit_target_encoded           
                          count       mean
unit                                      
cl                           33  18.225617
fl oz                      3228  16.505050
fluid ounce                 618  23.782834
g                          4205  26.868154
kg                          130  38.261735


C:\Users\nikk6\AppData\Local\Temp\ipykernel_21868\3239560314.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['unit_target_encoded'].fillna(global_mean, inplace=True)


In [19]:
# Cell 5 - load test, apply same transforms, and set unit_target_encoded using train mapping

# Load test (fallback to dummy if missing)
try:
    test_df = robust_read_csv(TEST_RAW)
    print("Loaded test.csv shape:", test_df.shape)
except FileNotFoundError:
    print("TEST_RAW not found. Creating dummy test_df for demonstration.")
    test_df = pd.DataFrame({
        'sample_id': [11111, 22222],
        'catalog_content': [
            "Item Name: Some Snack, 3 oz - 2 pack",
            "Item Name: Olive Oil, 500 ml bottle"
        ],
        'image_link': ['tlink1', 'tlink2'],
        # test may not have price; if it does, we'll keep it
    })
    print("Dummy test_df created.")

# Ensure required columns exist
for req in ['sample_id', 'catalog_content', 'image_link']:
    if req not in test_df.columns:
        test_df[req] = np.nan

# Apply feature extraction
temp = test_df['catalog_content'].apply(extract_item_size_and_unit).apply(pd.Series)
test_df['item_size'] = temp[0]
test_df['unit'] = temp[1]
test_df['pack_count'] = test_df['catalog_content'].apply(extract_pack_count_final_v2)

test_df['item_size'].fillna(1.0, inplace=True)
test_df['unit'].fillna('unit', inplace=True)
test_df['pack_count'].fillna(1.0, inplace=True)
test_df['total_quantity'] = test_df['item_size'] * test_df['pack_count']

# normalize unit strings the same way as train
test_df['unit'] = test_df['unit'].replace({
    'oz': 'ounce', 'o z': 'ounce', 'ounces': 'ounce',
    'milliliter': 'ml', 'milliliters': 'ml', 'mls': 'ml',
    'gram': 'g', 'grams': 'g', 'kilogram': 'kg', 'kilograms': 'kg',
    'pound': 'lb', 'pounds': 'lb', 'lbs': 'lb',
    'piece': 'pc', 'pieces': 'pc'
})

# Create unit_target_encoded for test using full-train mapping (non-CV)
if 'price' in train_df.columns:
    train_full_map = train_df.groupby('unit')['price'].mean()
    global_mean = train_df['price'].mean()
    test_df['unit_target_encoded'] = test_df['unit'].map(train_full_map)
    test_df['unit_target_encoded'].fillna(global_mean, inplace=True)
    print("unit_target_encoded mapped to test using train means. Global mean used for unseen units.")
else:
    test_df['unit_target_encoded'] = np.nan
    print("No price in train to build mapping; test unit_target_encoded set to NaN.")

display(test_df[['sample_id','catalog_content','item_size','unit','pack_count','total_quantity','unit_target_encoded']].head())


Loaded test.csv shape: (75000, 3)
unit_target_encoded mapped to test using train means. Global mean used for unseen units.


C:\Users\nikk6\AppData\Local\Temp\ipykernel_21868\2566242572.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df['item_size'].fillna(1.0, inplace=True)
C:\Users\nikk6\AppData\Local\Temp\ipykernel_21868\2566242572.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

,sample_id,catalog_content,item_size,unit,pack_count,total_quantity,unit_target_encoded
0,100179,Item Name: Rani 14-Spice Eshamaya's Mango Chut...,10.5,ounce,1.0,10.5,19.747364
1,245611,Item Name: Natural MILK TEA Flavoring extract ...,2.0,ounce,1.0,2.0,19.747364
2,146263,Item Name: Honey Filled Hard Candy - Bulk Pack...,2.0,lb,2.0,4.0,42.189786
3,95658,Item Name: Vlasic Snack'mm's Kosher Dill 16 Oz...,16.0,ounce,2.0,32.0,19.747364
4,36806,"Item Name: McCormick Culinary Vanilla Extract,...",32.0,fl oz,1.0,32.0,16.507048


In [20]:
# Cell 6 - save processed train & test with robust quoting and required columns

required_cols = ['sample_id','catalog_content','image_link','price','item_size','unit','pack_count','total_quantity','unit_target_encoded']

# Helper to select + create missing cols
def finalize_df_for_save(df, is_train=True):
    df = df.copy()
    # ensure all required exist (add missing ones as NaN)
    for c in required_cols:
        if c not in df.columns:
            df[c] = np.nan
    # ensure correct order
    df = df[required_cols]
    return df

train_to_save = finalize_df_for_save(train_df, is_train=True)
test_to_save  = finalize_df_for_save(test_df, is_train=False)

# Save with QUOTE_ALL so catalog_content with commas/newlines is preserved
train_to_save.to_csv(TRAIN_OUT, index=False, quoting=csv.QUOTE_ALL, encoding='utf-8')
test_to_save.to_csv(TEST_OUT, index=False, quoting=csv.QUOTE_ALL, encoding='utf-8')

print("Saved files:")
print(" -", TRAIN_OUT, " shape:", train_to_save.shape)
print(" -", TEST_OUT,  " shape:", test_to_save.shape)

# quick check: print first row of each saved df (in-memory)
print("\nTrain saved columns and dtypes:")
print(train_to_save.dtypes)
print("\nTest saved columns and dtypes:")
print(test_to_save.dtypes)

display(train_to_save.head(3))
display(test_to_save.head(3))


Saved files:
 - ..\data\processed\train_features_text.csv  shape: (75000, 9)
 - ..\data\processed\test_features_text.csv  shape: (75000, 9)

Train saved columns and dtypes:
sample_id                int64
catalog_content         object
image_link              object
price                  float64
item_size              float64
unit                    object
pack_count             float64
total_quantity         float64
unit_target_encoded    float64
dtype: object

Test saved columns and dtypes:
sample_id                int64
catalog_content         object
image_link              object
price                  float64
item_size              float64
unit                    object
pack_count             float64
total_quantity         float64
unit_target_encoded    float64
dtype: object


,sample_id,catalog_content,image_link,price,item_size,unit,pack_count,total_quantity,unit_target_encoded
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,12.0,ounce,6.0,72.0,19.839793
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,8.0,ounce,4.0,32.0,19.839793
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,1.9,ounce,1.0,1.9,19.674755


,sample_id,catalog_content,image_link,price,item_size,unit,pack_count,total_quantity,unit_target_encoded
0,100179,Item Name: Rani 14-Spice Eshamaya's Mango Chut...,https://m.media-amazon.com/images/I/71hoAn78AW...,NaN,10.5,ounce,1.0,10.5,19.747364
1,245611,Item Name: Natural MILK TEA Flavoring extract ...,https://m.media-amazon.com/images/I/61ex8NHCIj...,NaN,2.0,ounce,1.0,2.0,19.747364
2,146263,Item Name: Honey Filled Hard Candy - Bulk Pack...,https://m.media-amazon.com/images/I/61KCM61J8e...,NaN,2.0,lb,2.0,4.0,42.189786


In [11]:
test_df.shape


(75000, 8)